# Tomahawk G2 corrected 16 T — Kaggle Run 2/3

Attach the same input ZIP and the completed Tomahawk_G2_Run_1_checkpoint.zip. Enable Internet. Run All and save Run 2's checkpoint ZIP.

The notebook automatically finds Kaggle inputs and checks the full 96-plane MGRID SHA-256. It stops with a checkpoint before the 12-hour session cap. Download the ZIP under `/kaggle/working` even if a solve fails. PF limits here are exploratory ±50% bounds; numerical convergence alone is not hardware approval.


In [1]:
import hashlib
import importlib
import json
import os
import shutil
import signal
import subprocess
import sys
import time
import traceback
import zipfile
from pathlib import Path

RUN = 2
STAGE = "run_%d" % RUN
INPUT_NAME = "Tomahawk_G2_Corrected_Input.zip"
OUTPUT_NAME = "Tomahawk_G2_Run_%d_checkpoint.zip" % RUN
input_root = Path(os.environ.get("TOMAHAWK_INPUT_ROOT", "/kaggle/input"))
work_root = Path(os.environ.get("TOMAHAWK_WORK_ROOT", "/kaggle/working"))
work_root.mkdir(parents=True, exist_ok=True)
work = work_root / STAGE
work.mkdir(exist_ok=True)
start = time.monotonic()
deadline = start + 10*3600 + 45*60  # 75 minutes for Kaggle overhead and ZIP save
status = {"stage": RUN, "complete": False, "steps": [], "G2_approved": False}
problem = None

def digest(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(4*1024*1024), b""):
            h.update(block)
    return h.hexdigest()

def source_dir():
    archives = sorted(input_root.rglob(INPUT_NAME)) if input_root.exists() else []
    dirs = sorted(input_root.rglob("G2_INPUT_MANIFEST.json")) if input_root.exists() else []
    if len(archives) > 1 or (not archives and len(dirs) != 1):
        raise FileNotFoundError("Attach exactly one " + INPUT_NAME + " as Kaggle Input")
    dest = work_root / "tomahawk_g2_source"
    dest.mkdir(exist_ok=True)
    if archives:
        with zipfile.ZipFile(archives[0]) as z:
            names = z.namelist()
            if len(names) != len(set(names)) or "G2_INPUT_MANIFEST.json" not in names:
                raise ValueError("Input ZIP has duplicate files or no manifest")
            manifest = json.loads(z.read("G2_INPUT_MANIFEST.json"))
            expected = set(manifest["file_sha256"]) | {"G2_INPUT_MANIFEST.json"}
            if set(names) != expected:
                raise ValueError("Input ZIP member list does not match its manifest")
            for name in sorted(expected):
                with z.open(name) as src, (dest/name).open("wb") as target:
                    shutil.copyfileobj(src, target, 4*1024*1024)
    else:
        folder = dirs[0].parent
        shutil.copyfile(dirs[0], dest/"G2_INPUT_MANIFEST.json")
        manifest = json.loads(dirs[0].read_text())
        for name in manifest["file_sha256"]:
            source = folder/name
            if not source.is_file():
                raise FileNotFoundError(str(source))
            # Use the extracted Kaggle dataset directly; no 650 MB extra copy.
            link=dest/name
            if link.exists() or link.is_symlink():
                link.unlink()
            link.symlink_to(source)
    for name, wanted in manifest["file_sha256"].items():
        if digest(dest/name) != wanted:
            raise ValueError("SHA-256 mismatch for " + name)
    actual = digest(dest/"G2_INPUT_MANIFEST.json")
    if manifest["file_sha256"]["mgrid_tomahawk_16T_TF_PF_16FP_FULL_RECOMPUTED.nc"] != \
            "afb07089205aa3d2cdfa03d53b48298ae5092f23c0e70db72938c21f59a78013":
        raise ValueError("Expected the exact prior-run 96-plane MGRID")
    return dest, manifest, actual

def prior_stage(source_hash, engine_hash):
    if RUN == 1:
        return None
    filename = "Tomahawk_G2_Run_%d_checkpoint.zip" % (RUN-1)
    zips = sorted(input_root.rglob(filename))
    if len(zips) == 1:
        dest = work_root / "previous_g2_checkpoint"
        dest.mkdir(exist_ok=True)
        with zipfile.ZipFile(zips[0]) as z:
            for item in z.infolist():
                p = Path(item.filename)
                if item.is_dir():
                    continue
                if p.is_absolute() or ".." in p.parts:
                    raise ValueError("Unsafe previous checkpoint path")
                path = dest/p
                path.parent.mkdir(parents=True, exist_ok=True)
                with z.open(item) as src, path.open("wb") as dst:
                    shutil.copyfileobj(src,dst)
    elif len(zips) == 0:
        matches = [p for p in input_root.rglob("run_status.json")
                   if json.loads(p.read_text()).get("stage") == RUN-1]
        if len(matches) != 1:
            raise FileNotFoundError("Attach exactly one " + filename + " from completed Run %d"%(RUN-1))
        dest = matches[0].parent
    else:
        raise ValueError("Multiple previous checkpoints were attached")
    report = json.loads((dest/"run_status.json").read_text())
    if (report.get("stage"),report.get("complete"),report.get("source_manifest_sha256"),
        report.get("engine_sha256")) != (RUN-1,True,source_hash,engine_hash):
        raise ValueError("Prior run is incomplete or used another source/engine")
    previous = dest / ("steps/L10/free_equilibrium.h5" if RUN == 2
                       else "steps/L16/free_equilibrium.h5")
    if not previous.is_file():
        raise ValueError("Prior checkpoint lacks the completed equilibrium")
    return dest

def install_dependencies():
    if os.environ.get("TOMAHAWK_DRY_RUN") == "1":
        return
    required = [("numpy", "numpy"), ("scipy", "scipy"),
                ("desc", "desc-opt==0.17.3"), ("netCDF4", "netCDF4"),
                ("h5py", "h5py")]
    missing=[]
    for module, package in required:
        try:
            importlib.import_module(module)
        except ImportError:
            missing.append(package)
    if missing:
        print("Installing Kaggle dependencies:",missing,flush=True)
        subprocess.check_call([sys.executable,"-m","pip","install","--quiet",*missing])
    import numpy, scipy, desc
    if not desc.__version__.startswith("0.17.3"):
        raise RuntimeError("DESC 0.17.3 required, found " + desc.__version__)
    print("Python",sys.version.split()[0],"NumPy",numpy.__version__,
          "SciPy",scipy.__version__,"DESC",desc.__version__,flush=True)

def run_command(name, *args):
    remaining = deadline-time.monotonic()-600
    if remaining <= 0:
        raise TimeoutError("11-hour safety budget reached before "+name)
    log = work / (name+".log")
    cmd=[sys.executable,"-u",str(source/"g2_engine.py"),*map(str,args)]
    print("RUN",name,"time available (h)",round(remaining/3600,2),flush=True)
    t=time.monotonic()
    with log.open("w") as stream:
        proc=subprocess.Popen(cmd,stdout=stream,stderr=subprocess.STDOUT,
                              start_new_session=True)
        try:
            proc.wait(timeout=remaining)
        except subprocess.TimeoutExpired:
            os.killpg(proc.pid,signal.SIGTERM)
            try: proc.wait(timeout=30)
            except subprocess.TimeoutExpired:
                os.killpg(proc.pid,signal.SIGKILL);proc.wait()
            status["steps"].append({"name":name,"state":"TIME_BUDGET",
                                    "seconds":round(time.monotonic()-t,1)})
            raise TimeoutError(name+" ran out of Kaggle-safe time; checkpoint saved")
    status["steps"].append({"name":name,"state":"DONE" if proc.returncode==0 else "FAILED",
                            "exit_code":proc.returncode,"seconds":round(time.monotonic()-t,1)})
    print(log.read_text()[-5000:],flush=True)
    if proc.returncode:
        raise RuntimeError(name+" failed; its log is inside the checkpoint ZIP")

def solve_step(label,level,prior,scenario="baseline"):
    folder=work/"steps"/label
    folder.mkdir(parents=True,exist_ok=True)
    cmd=["solve","--input-dir",source,"--out-dir",folder,
         "--level",level,"--scenario",scenario]
    if prior:cmd.extend(["--prior",prior])
    run_command(label,*cmd)
    return folder/"free_equilibrium.h5"

def ballooning_step(label,eq_file):
    folder=work/"steps"/label
    folder.mkdir(parents=True,exist_ok=True)
    run_command(label,"stability","--input-dir",source,"--out-dir",folder,
                "--prior",eq_file)
    return json.loads((folder/"ideal_ballooning.json").read_text())

try:
    source, manifest, source_hash = source_dir()
    status["source_manifest_sha256"] = source_hash
    status["engine_sha256"] = manifest["file_sha256"]["g2_engine.py"]
    previous=prior_stage(source_hash,status["engine_sha256"])
    if RUN==1:
        run_command("exact_field_diagnosis","diagnose","--input-dir",source,
                    "--out-dir",work)
    if os.environ.get("TOMAHAWK_DRY_RUN") == "1":
        status["dry_run"] = True
    else:
        install_dependencies()
        if RUN==1:
            eq8=solve_step("L8","8,8,2",None)
            eq10=solve_step("L10","10,10,2",eq8)
        elif RUN==2:
            eq12=solve_step("L12","12,12,3",previous/"steps/L10/free_equilibrium.h5")
            eq16=solve_step("L16","16,16,4",eq12)
        else:
            eq18=solve_step("L18_baseline","18,18,4",
                            previous/"steps/L16/free_equilibrium.h5")
            stable=ballooning_step("baseline_ballooning",eq18)
            baseline=json.loads((eq18.parent/"metrics.json").read_text())
            if stable["ideal_ballooning_screen_pass"] and baseline["passes_conservative_numerical_gates"]:
                high16=solve_step("L16_high","16,16,4",eq18,"high")
                high18=solve_step("L18_high","18,18,4",high16,"high")
                highstable=ballooning_step("high_ballooning",high18)
                highmetrics=json.loads((high18.parent/"metrics.json").read_text())
                projection={"cores":10,"GWth_per_core":6.65,"usable_fraction":0.9400568,
                            "conversion_efficiency":0.4,"recirculating_GWe":3.0,
                            "conditional_net_GWe":10*6.65*0.9400568*0.4-3.0,
                            "numerical_boundary_force_gates_pass":highmetrics["passes_conservative_numerical_gates"],
                            "ideal_ballooning_screen_pass":highstable["ideal_ballooning_screen_pass"],
                            "hardware_qualified":False,"independent_equilibrium_comparison":False}
                (work/"conditional_power.json").write_text(json.dumps(projection,indent=2))
            else:
                (work/"high_pressure_gate.json").write_text(json.dumps({
                    "skipped":True,"reason":"Baseline finite-pressure G2 or ideal-ballooning gate did not pass",
                    "baseline_gates_pass":baseline["passes_conservative_numerical_gates"],
                    "baseline_ballooning_pass":stable["ideal_ballooning_screen_pass"]},indent=2))
        status["complete"] = True
except BaseException as exc:
    problem = exc
    status["error"] = str(exc)
    status["traceback"] = traceback.format_exc()
finally:
    status["elapsed_seconds"] = round(time.monotonic()-start,1)
    (work/"run_status.json").write_text(json.dumps(status,indent=2))
    output = work_root/OUTPUT_NAME
    with zipfile.ZipFile(output,"w",zipfile.ZIP_DEFLATED,compresslevel=5) as z:
        for file in sorted(work.rglob("*")):
            if file.is_file():
                z.write(file,file.relative_to(work))
    print("CHECKPOINT ZIP:",output,"bytes:",output.stat().st_size,
          "SHA-256:",digest(output),flush=True)
if problem is not None:
    raise RuntimeError("Run %d stopped. Download the checkpoint ZIP above; %s"%(RUN,problem)) from problem
if status.get("dry_run"):
    print("LOCAL INPUT DIAGNOSIS ONLY; NO DESC EQUILIBRIUM RUN")
else:
    print("DONE RUN",RUN,"— download",OUTPUT_NAME)


CHECKPOINT ZIP: /kaggle/working/Tomahawk_G2_Run_2_checkpoint.zip bytes: 582 SHA-256: f82fba990e7525ed0095a0379a0462006ba1074cf68cdd085973f86bbb05e42a


RuntimeError: Run 2 stopped. Download the checkpoint ZIP above; Attach exactly one Tomahawk_G2_Run_1_checkpoint.zip from completed Run 1